In [20]:
import seaborn as sns
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from cycler import cycler
from utils.metrics import calculate_mean_rocs

default_cycle = cycler(
    "linestyle",
    [
        "solid",
        "dotted",
        "dashdot",
        "dashed",
        (5, (10, 3)),
        (0, (3, 1, 1, 1, 1, 1)),
        (5, (10, 3)),
    ],
) + cycler(color=sns.color_palette()[:7])

line_styles = [
    "solid",
    "dotted",
    "dashdot",
    "dashed",
    (5, (10, 3)),
    (0, (3, 1, 1, 1, 1, 1)),
    (5, (10, 3)),
]

result_path = Path("..", "notebooks", "diagrams", "mrs_rocs")
result_path.mkdir(exist_ok=True)

In [21]:
data_set = "breast_cancer"
bias_type = "less_positive_class"
bias_strength = 0.1
method="mrs_step"
path = Path(f"../results/mrs_analysis/{method}/{data_set}/{bias_type}/{bias_strength}/mrs")

In [22]:
with open(path / "rocs/rocs.json") as file:
    roc_dict_list = json.load(file)
with open(path / "metadata.json") as file:
    metadata = json.load(file)

In [23]:
mean_roc_dict = calculate_mean_rocs(roc_dict_list)

In [24]:
plt.rc("")
plt.rc("axes", prop_cycle=default_cycle)
for hyperparameter, mean_roc in mean_roc_dict.items():
    for i, (fper, tper, std, deleted_elements) in enumerate(mean_roc):
        tpfrs_higher = np.minimum(tper + std, 1)
        tpfrs_lower = np.maximum(tper - std, 0)
        sns.lineplot(x=fper, y=tper, label=f"{deleted_elements} samples removed", linestyle=line_styles[i])
        plt.fill_between(fper, tpfrs_lower, tpfrs_higher, alpha=0.3)
    sns.lineplot(
        x=[0, 1],
        y=[0, 1],
        color="black",
        linestyle="--",
        linewidth=1,
    )
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.savefig(f"{result_path}/roc_{data_set}_{bias_strength}_{bias_type}_{hyperparameter}.pdf")
    plt.close()